In [2]:
import os

In [3]:
# --- 1. Install deps ---
!pip install sentencepiece tqdm

# --- 2. (Optional) Mount Google Drive to persist models ---
from google.colab import drive
drive.mount('/content/drive')

# You can choose where to save models
MODEL_DIR = "/content/drive/MyDrive/shakespeare_model"
os.makedirs(MODEL_DIR, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ----------------------------
# 1) Read book files
# ----------------------------
def read_text_files(dir_path: str):
    texts = []
    if not os.path.isdir(dir_path):
        return texts
    for fname in os.listdir(dir_path):
        if fname.endswith(".txt"):
            with open(os.path.join(dir_path, fname), "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        texts.append(line)
    return texts

# ----------------------------
# 2) Train SentencePiece BPE
# ----------------------------
def train_bpe(all_texts, model_prefix="bpe", vocab_size=8000):
    combined_file = f"{model_prefix}_combined.txt"
    with open(combined_file, "w", encoding="utf-8") as f:
        for t in all_texts:
            f.write(t.replace("\n", " ") + "\n")
    spm.SentencePieceTrainer.Train(
        input=combined_file,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        model_type="bpe",
        character_coverage=0.9995,
        unk_id=0, pad_id=1, bos_id=2, eos_id=3
    )
    sp = spm.SentencePieceProcessor()
    sp.load(f"{model_prefix}.model")
    return sp

# ----------------------------
# 3) Encode sentences -> token IDs
# ----------------------------
def encode_sentence(sentence, sp, add_bos=False, add_eos=False):
    ids = sp.encode(sentence, out_type=int)
    if add_bos:
        ids = [sp.bos_id()] + ids
    if add_eos:
        ids = ids + [sp.eos_id()]
    return ids

# ----------------------------
# 4) Dataset + DataLoader
# ----------------------------
class ParallelTextDataset(Dataset):
    def __init__(self, modern_sentences, shake_sentences, sp):
        assert len(modern_sentences) == len(shake_sentences), "Parallel corpora must be aligned!"
        self.modern = modern_sentences
        self.shake = shake_sentences
        self.sp = sp

    def __len__(self):
        return len(self.modern)

    def __getitem__(self, idx):
        src_ids = encode_sentence(self.modern[idx], self.sp, add_bos=False, add_eos=True)
        tgt_in_ids = encode_sentence(self.shake[idx], self.sp, add_bos=True, add_eos=False)
        tgt_out_ids = encode_sentence(self.shake[idx], self.sp, add_bos=False, add_eos=True)
        return torch.tensor(src_ids), torch.tensor(tgt_in_ids), torch.tensor(tgt_out_ids)

def pad_sequence(seqs, pad_id):
    max_len = max(s.size(0) for s in seqs)
    out = torch.full((len(seqs), max_len), pad_id, dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, :s.size(0)] = s
    return out

def collate_fn(batch):
    pad_id = 1
    src_list, tgt_in_list, tgt_out_list = zip(*batch)
    src = pad_sequence(src_list, pad_id)
    tgt_in = pad_sequence(tgt_in_list, pad_id)
    tgt_out = pad_sequence(tgt_out_list, pad_id)
    return src, tgt_in, tgt_out

# ----------------------------
# 5) Seq2Seq Model
# ----------------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, pad_id, num_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.rnn = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers,
                          batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src_ids):
        src_ids = src_ids.to(self.embedding.weight.device)   # 🔧 ensure same device
        emb = self.dropout(self.embedding(src_ids))
        outputs, hidden = self.rnn(emb)
        H = outputs.size(2) // 2
        outputs = outputs[:, :, :H] + outputs[:, :, H:]
        L2, B, Hh = hidden.size()
        hidden = hidden[0:L2:2] + hidden[1:L2:2]
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, dec_hidden, enc_outputs):
        B, Ls, H = enc_outputs.shape
        dec = dec_hidden.unsqueeze(1).repeat(1, Ls, 1)
        energy = torch.tanh(self.attn(torch.cat([dec, enc_outputs], dim=2)))
        scores = self.v(energy).squeeze(2)
        return F.softmax(scores, dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, pad_id, attention, num_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = attention
        self.rnn = nn.GRU(emb_dim + hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim * 2, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, enc_outputs):
        input_token = input_token.to(self.embedding.weight.device)   # 🔧 ensure same device
        emb = self.dropout(self.embedding(input_token)).unsqueeze(1)
        dec_hidden_last = hidden[-1]
        attn_weights = self.attention(dec_hidden_last, enc_outputs)
        context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs)
        rnn_in = torch.cat([emb, context], dim=2)
        output, hidden = self.rnn(rnn_in, hidden)
        logits = self.fc_out(torch.cat([output, context], dim=2)).squeeze(1)
        return logits, hidden, attn_weights

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, pad_id, bos_id, eos_id, sp):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_id = pad_id
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.sp = sp

    def forward(self, src_ids, tgt_in_ids, tgt_out_ids, teacher_forcing_ratio=0.5):
        B, Lt = tgt_out_ids.shape
        enc_outputs, hidden = self.encoder(src_ids)
        logits_seq = []
        inp = tgt_in_ids[:, 0]
        for t in range(Lt):
            logits, hidden, _ = self.decoder(inp, hidden, enc_outputs)
            logits_seq.append(logits.unsqueeze(1))
            teacher = (torch.rand(1).item() < teacher_forcing_ratio)
            if t + 1 < Lt:
                inp = tgt_in_ids[:, t+1] if teacher else logits.argmax(dim=1)
        return torch.cat(logits_seq, dim=1)

    @torch.no_grad()
    def beam_search(self, src_ids, beam_width=3, max_len=60):
        enc_outputs, hidden = self.encoder(src_ids)
        B = src_ids.size(0)
        beams = [( [self.bos_id], hidden, 0 )]  # (sequence, hidden, score)

        for _ in range(max_len):
            new_beams = []
            for seq, h, score in beams:
                inp = torch.tensor([seq[-1]], device=src_ids.device)
                logits, h_next, _ = self.decoder(inp, h, enc_outputs)
                probs = F.log_softmax(logits, dim=1).squeeze(0)
                topk = torch.topk(probs, beam_width)
                for idx, val in zip(topk.indices.tolist(), topk.values.tolist()):
                    new_seq = seq + [idx]
                    new_score = score + val
                    new_beams.append((new_seq, h_next, new_score))
            beams = sorted(new_beams, key=lambda x: x[2], reverse=True)[:beam_width]
            if all(self.eos_id in seq for seq,_,_ in beams):
                break

        best_seq = beams[0][0]
        if self.eos_id in best_seq:
            best_seq = best_seq[:best_seq.index(self.eos_id)]
        return self.sp.decode(best_seq[1:])  # remove BOS

# ----------------------------
# 6) Training
# ----------------------------
def train_model(model, dataloader, sp, epochs=10, lr=1e-3, device=None):
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    pad_id = sp.pad_id()
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id, label_smoothing=0.1)

    for epoch in range(1, epochs+1):
        model.train()
        epoch_loss = 0.0
        teacher_forcing_ratio = max(0.5 * (0.95 ** epoch), 0.1)  # decays each epoch
        pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{epochs}")
        for src, tgt_in, tgt_out in pbar:
            src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
            optimizer.zero_grad()
            logits = model(src, tgt_in, tgt_out, teacher_forcing_ratio)
            B, Lt, V = logits.shape
            loss = criterion(logits.reshape(B*Lt, V), tgt_out.reshape(B*Lt))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.2f}"})
        print(f"Epoch {epoch}/{epochs}, Avg Loss: {epoch_loss/len(dataloader):.4f}")
    print("Training complete.")

# ----------------------------
# 7) Main
# ----------------------------
def main(modern_dir="data/modern", shake_dir="data/shakespeare",
         batch_size=32, epochs=10, retrain=False, vocab_size=8000, emb_dim=128, hidden_dim=128):

    modern_texts = read_text_files(modern_dir)
    shake_texts  = read_text_files(shake_dir)
    all_texts = modern_texts + shake_texts
    if not all_texts:
        raise RuntimeError("No text files found in modern/ or shakespeare/")

    sp = train_bpe(all_texts, vocab_size=vocab_size)
    pad_id, bos_id, eos_id = sp.pad_id(), sp.bos_id(), sp.eos_id()
    vocab_size = sp.get_piece_size()

    dataset = ParallelTextDataset(modern_texts, shake_texts, sp)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    enc = Encoder(vocab_size=vocab_size, emb_dim=emb_dim, hidden_dim=hidden_dim, pad_id=pad_id)
    attn = Attention(hidden_dim)
    dec = Decoder(vocab_size=vocab_size, emb_dim=emb_dim, hidden_dim=hidden_dim, pad_id=pad_id, attention=attn)
    model = Seq2Seq(enc, dec, pad_id, bos_id, eos_id, sp)

    ckpt = "shakespeare_model.pth"
    if (not retrain) and os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location="cpu"))
        model.eval()
        print(f"Loaded pretrained model from {ckpt}")
    else:
        print("Starting training...")
        train_model(model, dataloader, sp, epochs=epochs)
        torch.save(model.state_dict(), ckpt)
        print(f"Model saved to {ckpt}")

    example_sentence = "Where are you going?"
    src_ids = torch.tensor(encode_sentence(example_sentence, sp, add_eos=True)).unsqueeze(0)
    pred_text = model.beam_search(src_ids)
    print("Original:", example_sentence)
    print("Predicted Shakespearean:", pred_text)

    return sp, dataloader, model

# ----------------------------
# 8) Run
# ----------------------------
if __name__ == "__main__":
  sp, dataloader, model = main(
      batch_size=16,
      epochs=10,
      emb_dim=128,
      hidden_dim=128,
      retrain=True   # <-- make sure retrain flag is used
  )


Starting training...


Epoch 1/10: 100%|██████████| 330/330 [02:00<00:00,  2.74it/s, loss=6.75]


Epoch 1/10, Avg Loss: 6.9795


Epoch 2/10: 100%|██████████| 330/330 [01:58<00:00,  2.79it/s, loss=5.90]


Epoch 2/10, Avg Loss: 6.7208


...

Epoch 8/10, Avg Loss: 6.0309


Epoch 9/10: 100%|██████████| 330/330 [01:58<00:00,  2.79it/s, loss=5.76]


Epoch 9/10, Avg Loss: 5.9407


Epoch 10/10: 100%|██████████| 330/330 [01:57<00:00,  2.80it/s, loss=5.18]


Epoch 10/10, Avg Loss: 5.8581
Training complete.
Model saved to shakespeare_model.pth
Original: Where are you going?
Predicted Shakespearean: We will not my wife.
